# Music to MIDI — Web UI Edition (Colab GPU)

Notebook này khởi chạy **giao diện Web UI đầy đủ của `music-to-midi`** (`web/` + FastAPI `src.web_api`) trên Google Colab GPU, thay thế cho giao diện Gradio cũ.

### Điểm mới & khắc phục lỗi Colab hiện tại
- **Môi trường Python 3.11 cô lập (`uv venv`)**: Google Colab hiện dùng Python 3.13 + NumPy 2.4 + PyTorch 2.14, gây xung đột với các package khóa phiên bản cũ trong `colab_notebook.ipynb`. Notebook này tạo `venv` Python 3.11 riêng tại `/content/music-to-midi/venv` với bộ thư viện chuẩn CUDA 12.8 (`torch==2.7.0`, `numpy==1.26.4`, `onnxruntime-gpu==1.23.2`, `audio-separator==0.44.1`, `transkun==2.0.1`, `muscriptor==0.4.0`, `aria-amt`), hoàn toàn không đụng chạm Python hệ thống của Colab.
- **Giao diện Web UI đầy đủ tính năng**: Quản lý dự án, 7 chế độ chuyển đổi, waveform đa track, mixer, SoundFont preview/render, xuất MIDI tách track & xuất Sheet Music (MuseScore Studio 4), cập nhật tiến độ real-time qua SSE.
- **Same-origin qua Cloudflare Quick Tunnel / Colab Proxy**: Phục vụ cả Frontend và API trên cùng 1 cổng (`7860`) với `/runtime-config.json` động, không gặp lỗi CORS hay kết nối nhầm `127.0.0.1`.

### Thứ tự chạy
1. **Cell 1**: Kiểm tra GPU NVIDIA (khuyến nghị `T4` trở lên).
2. **Cell 2**: Đồng bộ repo và cài đặt môi trường cô lập Python 3.11 + CUDA 12.8.
3. **Cell 3**: Tải & kiểm tra checksum các model AI theo preset bạn chọn (tùy chọn bật `HF_TOKEN` cho MuScriptor gated).
4. **Cell 4**: Khởi chạy Web UI và mở link công khai (`https://*.trycloudflare.com`).


In [ ]:
#@title 1️⃣ Kiểm tra GPU & Môi trường Colab
import shutil
import subprocess
import sys
from datetime import datetime

import torch


def log(message=""):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)


log("Kiểm tra phần cứng và môi trường Colab")
log(f"Colab Kernel Python: {sys.version.split()[0]}")
log(f"Host PyTorch: {torch.__version__} (CUDA {torch.version.cuda})")
log(f"uv binary: {shutil.which('uv') or 'NOT FOUND'}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "Không phát hiện GPU NVIDIA. Trong Colab hãy chọn Runtime -> Change runtime type -> T4 GPU."
    )

device_index = torch.cuda.current_device()
props = torch.cuda.get_device_properties(device_index)
log(f"GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.2f} GB | SM: {props.multi_processor_count}")

smi = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    text=True,
).strip()
log(f"nvidia-smi: {smi}")
log("Kiểm tra GPU hoàn tất!")


In [ ]:
#@title 2️⃣ Đồng bộ mã nguồn & Cài đặt môi trường Python 3.11 + CUDA 12.8
import os
import shlex
import shutil
import subprocess
from datetime import datetime
from pathlib import Path

REPO_URL = "https://github.com/mason369/music-to-midi.git"  # @param {type:"string"}
BRANCH = "master"  # @param {type:"string"}
USE_EXISTING_WORKSPACE = True  # @param {type:"boolean"}
SYNC_GIT_REMOTE = False  # @param {type:"boolean"}

REPO_DIR = Path("/content/music-to-midi")
VENV_DIR = REPO_DIR / "venv"
VENV_PYTHON = VENV_DIR / "bin" / "python"

COLAB_SERVER_FALLBACK = '"""Single-origin FastAPI + Web frontend launcher for Google Colab."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport logging\nimport os\nfrom pathlib import Path\nfrom typing import Iterable\nfrom urllib.parse import urlsplit\n\nif "matplotlib_inline" in os.environ.get("MPLBACKEND", ""):\n    os.environ["MPLBACKEND"] = "Agg"\n\nimport uvicorn\nfrom starlette.types import ASGIApp, Message, Receive, Scope, Send\n\nfrom src.gui.web.server_runtime import configure_uvicorn_websocket_protocol\nfrom src.utils.runtime_paths import get_resource_path\nfrom src.web_api.app import create_app\nfrom src.web_contract import API_VERSION\n\nDEFAULT_HOST = "0.0.0.0"\nDEFAULT_PORT = 7860\nDEFAULT_RETENTION_DAYS = 30\nDEFAULT_RETENTION_MAX_JOBS = 200\n_LOOPBACK_HOSTS = {"0.0.0.0", "::", ""}\n_SECURITY_HEADERS: tuple[tuple[bytes, bytes], ...] = (\n    (b"x-content-type-options", b"nosniff"),\n    (b"referrer-policy", b"no-referrer"),\n    (\n        b"permissions-policy",\n        b"camera=(), microphone=(), geolocation=(), payment=(), usb=()",\n    ),\n)\n\n\ndef _header_map(raw_headers: Iterable[tuple[bytes, bytes]]) -> dict[str, str]:\n    headers: dict[str, str] = {}\n    for name, value in raw_headers:\n        key = name.decode("latin-1").strip().lower()\n        val = value.decode("latin-1").strip()\n        if key and val and key not in headers:\n            headers[key] = val\n    return headers\n\n\ndef _first_forwarded_token(value: str) -> str:\n    return value.split(",", 1)[0].strip()\n\n\ndef _validated_host_header(candidate: str) -> tuple[str, str] | None:\n    """Validate a Host/X-Forwarded-Host header and return (netloc, hostname) or None."""\n\n    value = str(candidate).strip()\n    if not value or any(ch.isspace() for ch in value):\n        return None\n    try:\n        parsed = urlsplit(f"http://{value}")\n        port = parsed.port\n    except ValueError:\n        return None\n    if (\n        not parsed.hostname\n        or parsed.path\n        or parsed.query\n        or parsed.fragment\n        or parsed.username is not None\n        or parsed.password is not None\n    ):\n        return None\n    if port is not None and not 1 <= port <= 65535:\n        return None\n    return parsed.netloc, parsed.hostname\n\n\ndef resolve_request_origin(\n    raw_headers: Iterable[tuple[bytes, bytes]],\n    *,\n    fallback_scheme: str = "http",\n    fallback_host: str = "127.0.0.1",\n    fallback_port: int = DEFAULT_PORT,\n) -> str:\n    """Resolve and validate the browser-visible origin from request/proxy headers."""\n\n    headers = _header_map(raw_headers)\n    forwarded_host = _first_forwarded_token(headers.get("x-forwarded-host", ""))\n    parsed_host = _validated_host_header(forwarded_host) or _validated_host_header(\n        headers.get("host", "")\n    )\n    if parsed_host is None:\n        safe_host = "127.0.0.1" if fallback_host in _LOOPBACK_HOSTS else fallback_host\n        fallback_candidate = (\n            f"[{safe_host}]:{fallback_port}"\n            if ":" in safe_host and not safe_host.startswith("[")\n            else f"{safe_host}:{fallback_port}"\n        )\n        parsed_host = _validated_host_header(fallback_candidate) or (\n            f"127.0.0.1:{DEFAULT_PORT}",\n            "127.0.0.1",\n        )\n\n    netloc, hostname = parsed_host\n    forwarded_proto = _first_forwarded_token(headers.get("x-forwarded-proto", "")).lower()\n    if forwarded_proto in {"http", "https"}:\n        scheme = forwarded_proto\n    elif hostname.lower().endswith(".trycloudflare.com"):\n        scheme = "https"\n    elif fallback_scheme in {"http", "https"}:\n        scheme = fallback_scheme\n    else:\n        scheme = "http"\n    return f"{scheme}://{netloc}".rstrip("/")\n\n\ndef build_runtime_config(origin: str) -> dict[str, object]:\n    """Return same-origin runtime-config.json payload for the standalone Web UI."""\n\n    normalized = origin.rstrip("/")\n    return {\n        "managed": True,\n        "frontend_url": normalized,\n        "backend_url": normalized,\n        "expected_api_version": API_VERSION,\n    }\n\n\nclass ColabRuntimeConfigMiddleware:\n    """Serve dynamic /runtime-config.json and baseline security headers."""\n\n    def __init__(\n        self,\n        app: ASGIApp,\n        *,\n        fallback_host: str = "127.0.0.1",\n        fallback_port: int = DEFAULT_PORT,\n    ) -> None:\n        self.app = app\n        self._fallback_host = fallback_host\n        self._fallback_port = fallback_port\n\n    async def __call__(self, scope: Scope, receive: Receive, send: Send) -> None:\n        if scope.get("type") != "http":\n            await self.app(scope, receive, send)\n            return\n\n        method = str(scope.get("method") or "GET").upper()\n        if scope.get("path") == "/runtime-config.json" and method in {"GET", "HEAD"}:\n            origin = resolve_request_origin(\n                scope.get("headers") or (),\n                fallback_scheme=str(scope.get("scheme") or "http"),\n                fallback_host=self._fallback_host,\n                fallback_port=self._fallback_port,\n            )\n            body = json.dumps(build_runtime_config(origin), ensure_ascii=False).encode("utf-8")\n            headers = [\n                (b"content-type", b"application/json; charset=utf-8"),\n                (b"cache-control", b"no-store"),\n                (b"content-length", str(len(body)).encode("ascii")),\n                *_SECURITY_HEADERS,\n            ]\n            await send({"type": "http.response.start", "status": 200, "headers": headers})\n            await send(\n                {\n                    "type": "http.response.body",\n                    "body": b"" if method == "HEAD" else body,\n                }\n            )\n            return\n\n        async def send_with_security_headers(message: Message) -> None:\n            if message.get("type") == "http.response.start":\n                raw = list(message.get("headers") or ())\n                existing = {name.lower() for name, _ in raw}\n                for header_name, header_value in _SECURITY_HEADERS:\n                    if header_name not in existing:\n                        raw.append((header_name, header_value))\n                message = {**message, "headers": raw}\n            await send(message)\n\n        await self.app(scope, receive, send_with_security_headers)\n\n\ndef create_colab_app(\n    *,\n    data_root: str | Path | None = None,\n    frontend_root: str | Path | None = None,\n    cors_origins: list[str] | None = None,\n    max_upload_bytes: int | None = None,\n    retention_days: int = DEFAULT_RETENTION_DAYS,\n    retention_max_jobs: int = DEFAULT_RETENTION_MAX_JOBS,\n    fallback_host: str = "127.0.0.1",\n    fallback_port: int = DEFAULT_PORT,\n) -> ASGIApp:\n    """Create the single-origin FastAPI + Web UI ASGI application for Colab."""\n\n    resolved_frontend = Path(frontend_root or get_resource_path("web")).resolve()\n    app = create_app(\n        data_root=data_root,\n        cors_origins=cors_origins,\n        frontend_root=resolved_frontend,\n        max_upload_bytes=max_upload_bytes,\n        retention_days=retention_days,\n        retention_max_jobs=retention_max_jobs,\n    )\n    return ColabRuntimeConfigMiddleware(\n        app,\n        fallback_host=fallback_host,\n        fallback_port=fallback_port,\n    )\n\n\ndef _parser() -> argparse.ArgumentParser:\n    parser = argparse.ArgumentParser(\n        description="Run the Music to MIDI Web UI + Inference API for Colab"\n    )\n    parser.add_argument("--host", default=DEFAULT_HOST)\n    parser.add_argument("--port", type=int, default=DEFAULT_PORT)\n    parser.add_argument("--data-dir", default=None)\n    parser.add_argument("--frontend-dir", default=None)\n    parser.add_argument(\n        "--cors-origin",\n        action="append",\n        default=None,\n        help="Optional explicit CORS origin; omit for single-origin mode",\n    )\n    parser.add_argument("--log-level", default="info")\n    parser.add_argument("--retention-days", type=int, default=DEFAULT_RETENTION_DAYS)\n    parser.add_argument("--retention-max-jobs", type=int, default=DEFAULT_RETENTION_MAX_JOBS)\n    return parser\n\n\ndef main(argv: list[str] | None = None) -> int:\n    args = _parser().parse_args(argv)\n    logging.basicConfig(\n        level=getattr(logging, str(args.log_level).upper(), logging.INFO),\n        format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",\n    )\n    app = create_colab_app(\n        data_root=args.data_dir,\n        frontend_root=args.frontend_dir,\n        cors_origins=args.cors_origin,\n        retention_days=args.retention_days,\n        retention_max_jobs=args.retention_max_jobs,\n        fallback_host=args.host,\n        fallback_port=args.port,\n    )\n    websocket_protocol = configure_uvicorn_websocket_protocol()\n    uvicorn.run(\n        app,\n        host=args.host,\n        port=args.port,\n        log_level=str(args.log_level).lower(),\n        ws=websocket_protocol,\n        proxy_headers=True,\n        forwarded_allow_ips="127.0.0.1,::1",\n    )\n    return 0\n\n\n__all__ = [\n    "ColabRuntimeConfigMiddleware",\n    "build_runtime_config",\n    "create_colab_app",\n    "main",\n    "resolve_request_origin",\n]\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'


def log(message=""):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)


def run_cmd(command, cwd=None):
    log(f"CMD: {command}")
    proc = subprocess.Popen(
        command,
        cwd=str(cwd) if cwd else None,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"Lệnh thất bại (exit={code}): {command}")


if not REPO_DIR.exists():
    run_cmd(f"git clone --branch {shlex.quote(BRANCH)} --single-branch {shlex.quote(REPO_URL)} {shlex.quote(str(REPO_DIR))}")
elif not USE_EXISTING_WORKSPACE:
    shutil.rmtree(REPO_DIR)
    run_cmd(f"git clone --branch {shlex.quote(BRANCH)} --single-branch {shlex.quote(REPO_URL)} {shlex.quote(str(REPO_DIR))}")
else:
    log(f"Sử dụng mã nguồn hiện có tại {REPO_DIR}")
    if SYNC_GIT_REMOTE and (REPO_DIR / ".git").is_dir():
        dirty = subprocess.check_output(
            ["git", "-C", str(REPO_DIR), "status", "--porcelain", "--untracked-files=no"],
            text=True,
        ).strip()
        if dirty:
            raise RuntimeError(f"Workspace đang có thay đổi chưa commit, từ chối pull:\n{dirty}")
        run_cmd(f"git -C {shlex.quote(str(REPO_DIR))} pull --ff-only origin {shlex.quote(BRANCH)}")

os.chdir(REPO_DIR)
colab_server_path = REPO_DIR / "src" / "web_api" / "colab_server.py"
if not colab_server_path.is_file():
    colab_server_path.parent.mkdir(parents=True, exist_ok=True)
    colab_server_path.write_text(COLAB_SERVER_FALLBACK, encoding="utf-8")
    log(f"Đã tạo {colab_server_path} cho môi trường Colab standalone")

if shutil.which("uv") is None:
    run_cmd("python3 -m pip install -q uv")

log("Cài đặt gói hệ thống (FFmpeg, FluidSynth, audio/GL libs) và cloudflared...")
run_cmd(
    "apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
    "ffmpeg fluidsynth libasound2t64 libegl1 libfontconfig1 libgl1 libglib2.0-0t64 "
    "libgomp1 libportaudio2 libsndfile1"
)
if shutil.which("cloudflared") is None:
    run_cmd(
        "curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
        "-o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared"
    )

if not VENV_PYTHON.is_file():
    log(f"Tạo môi trường cô lập Python 3.11 tại {VENV_DIR}...")
    run_cmd(f"uv venv --python 3.11 {shlex.quote(str(VENV_DIR))}")

py = shlex.quote(str(VENV_PYTHON))
log("Cài đặt PyTorch 2.7.0 + CUDA 12.8 và toàn bộ dependencies...")
run_cmd(f"uv pip install --python {py} --upgrade pip==25.2 setuptools==80.10.2 wheel==0.45.1")
run_cmd(
    f"uv pip install --python {py} torch==2.7.0 torchaudio==2.7.0 torchvision==0.22.0 "
    "--index-url https://download.pytorch.org/whl/cu128"
)

excluded_prefixes = (
    "torch==",
    "torchaudio==",
    "torchvision==",
    "numpy",
    "python-rtmidi",
    "PyQt6",
    "onnxruntime",
)
filtered_requirements = [
    line
    for line in (REPO_DIR / "requirements.txt").read_text(encoding="utf-8").splitlines()
    if not any(line.strip().startswith(prefix) for prefix in excluded_prefixes)
]
filtered_req_path = Path("/tmp/requirements-colab-webui.txt")
filtered_req_path.write_text("\n".join(filtered_requirements) + "\n", encoding="utf-8")

run_cmd(f"uv pip install --python {py} numpy==1.26.4")
run_cmd(f"uv pip install --python {py} -c docker/backend-constraints.txt -r {shlex.quote(str(filtered_req_path))}")
run_cmd(
    f"uv pip install --python {py} -c docker/backend-constraints.txt "
    "beartype==0.18.5 diffq-fixed==0.2.4 julius==0.2.7 ml_collections==1.1.0 "
    "onnx-weekly==1.21.0.dev20260223 onnx2torch-py313==1.6.0 onnxruntime-gpu==1.23.2 "
    "pydub==0.25.1 resampy==0.4.3 rotary-embedding-torch==0.6.5 samplerate==0.1.0 six==1.17.0"
)
run_cmd(f"uv pip install --python {py} --no-deps audio-separator==0.44.1")

muscriptor_req = subprocess.check_output(
    [str(VENV_PYTHON), "-c", "from src.utils.muscriptor_source_identity import MUSCRIPTOR_SOURCE_REQUIREMENT; print(MUSCRIPTOR_SOURCE_REQUIREMENT)"],
    cwd=str(REPO_DIR),
    text=True,
).strip()
run_cmd(f"uv pip install --python {py} --no-deps {shlex.quote(muscriptor_req)}")
run_cmd(
    f"uv pip install --python {py} --no-deps "
    "'aria-amt @ https://github.com/EleutherAI/aria-amt/archive/a1ab73fc901d1759ec3bc173c146b3c6a3040261.zip'"
)

log("Kiểm chứng runtime và MuseScore Studio 4...")
run_cmd(f"{py} -m src.utils.source_runtime", cwd=REPO_DIR)
run_cmd(f"{py} -m src.utils.aria_amt_runtime", cwd=REPO_DIR)
run_cmd(f"{py} download_musescore_runtime.py", cwd=REPO_DIR)
run_cmd(
    f"{py} -c \"import torch, onnxruntime as ort; "
    "assert torch.cuda.is_available(), 'CUDA unavailable'; "
    "providers = ort.get_available_providers(); "
    "assert 'CUDAExecutionProvider' in providers, providers; "
    "print('Verified CUDA + ORT:', torch.__version__, torch.version.cuda, providers)\"",
    cwd=REPO_DIR,
)
log("Cài đặt môi trường Web UI hoàn tất!")


In [ ]:
#@title 3️⃣ Chuẩn bị Model Weights & (Tùy chọn) MuScriptor gated
import os
import subprocess
from pathlib import Path

MODEL_PRESET = "recommended"  # @param ["recommended", "all_public", "all_including_muscriptor", "minimal_yourmt3_only", "custom"]
CUSTOM_PROFILES = ""  # @param {type:"string"}
ENABLE_MUSCRIPTOR = False  # @param {type:"boolean"}

REPO_DIR = Path("/content/music-to-midi")
VENV_PYTHON = REPO_DIR / "venv" / "bin" / "python"
PROFILE_FILE = REPO_DIR / ".colab_enabled_profiles"

PRESET_MAP = {
    "minimal_yourmt3_only": [
        "yourmt3:yptf_moe_multi_ps",
    ],
    "recommended": [
        "yourmt3:yptf_moe_multi_ps",
        "piano_transkun",
        "piano_bytedance_pedal",
        "vocal_split",
        "six_stem_split",
    ],
    "all_public": [
        "yourmt3:ymt3_plus",
        "yourmt3:yptf_single_nops",
        "yourmt3:yptf_multi_ps",
        "yourmt3:yptf_moe_multi_nops",
        "yourmt3:yptf_moe_multi_ps",
        "miros",
        "piano_transkun",
        "piano_transkun_v2_aug",
        "piano_aria_amt",
        "piano_bytedance_pedal",
        "vocal_split",
        "six_stem_split",
    ],
    "all_including_muscriptor": [
        "yourmt3:ymt3_plus",
        "yourmt3:yptf_single_nops",
        "yourmt3:yptf_multi_ps",
        "yourmt3:yptf_moe_multi_nops",
        "yourmt3:yptf_moe_multi_ps",
        "miros",
        "muscriptor",
        "muscriptor:medium",
        "muscriptor:small",
        "piano_transkun",
        "piano_transkun_v2_aug",
        "piano_aria_amt",
        "piano_bytedance_pedal",
        "vocal_split",
        "six_stem_split",
    ],
}

if MODEL_PRESET == "custom":
    selected_profiles = [item.strip() for item in CUSTOM_PROFILES.split(",") if item.strip()]
    if not selected_profiles:
        raise ValueError("Hãy nhập danh sách profile vào CUSTOM_PROFILES khi chọn preset 'custom'.")
else:
    selected_profiles = list(PRESET_MAP[MODEL_PRESET])

child_env = os.environ.copy()
child_env["MPLBACKEND"] = "Agg"
child_env["WANDB_MODE"] = "disabled"
child_env["TF_CPP_MIN_LOG_LEVEL"] = "3"
child_env["MPLCONFIGDIR"] = "/tmp/matplotlib"
child_env["NUMBA_CACHE_DIR"] = "/tmp/numba_cache"

uses_muscriptor = ENABLE_MUSCRIPTOR or any(p.startswith("muscriptor") for p in selected_profiles)
if uses_muscriptor:
    hf_token = child_env.get("HF_TOKEN", "").strip()
    if not hf_token:
        try:
            from google.colab import userdata

            hf_token = str(userdata.get("HF_TOKEN") or "").strip()
        except Exception:
            hf_token = ""
    if not hf_token:
        raise RuntimeError(
            "MuScriptor yêu cầu HF_TOKEN có quyền đọc Small/Medium/Large trong Colab Secrets."
        )
    child_env["HF_TOKEN"] = hf_token
    subprocess.check_call(
        [
            str(VENV_PYTHON),
            "-c",
            "import os; from huggingface_hub import login; "
            "from src.utils.muscriptor_downloader import preflight_muscriptor_download_access; "
            "login(token=os.environ['HF_TOKEN'], add_to_git_credential=False); "
            "preflight_muscriptor_download_access()",
        ],
        cwd=str(REPO_DIR),
        env=child_env,
    )
    if ENABLE_MUSCRIPTOR:
        for muscriptor_id in ("muscriptor", "muscriptor:medium", "muscriptor:small"):
            if muscriptor_id not in selected_profiles:
                selected_profiles.append(muscriptor_id)

profiles_arg = ",".join(dict.fromkeys(selected_profiles))
print(f"Đang chuẩn bị model profiles: {profiles_arg}", flush=True)
proc = subprocess.Popen(
    [str(VENV_PYTHON), "-m", "src.model_profiles", "prepare", "--profiles", profiles_arg],
    cwd=str(REPO_DIR),
    env=child_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line.rstrip(), flush=True)
if proc.wait() != 0:
    raise RuntimeError("Chuẩn bị model profiles thất bại.")

PROFILE_FILE.write_text(profiles_arg, encoding="utf-8")
print(f"Đã lưu cấu hình profiles vào {PROFILE_FILE}", flush=True)


In [ ]:
#@title 4️⃣ Khởi chạy Web UI (FastAPI + web/ Frontend + Tunnel)
import json
import os
import re
import shutil
import subprocess
import time
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

PORT = 7860  # @param {type:"integer"}
PUBLIC_TUNNEL = "cloudflared"  # @param ["cloudflared", "colab_proxy_only"]
KEEP_BLOCKING = True  # @param {type:"boolean"}

REPO_DIR = Path("/content/music-to-midi")
VENV_PYTHON = REPO_DIR / "venv" / "bin" / "python"
DATA_DIR = REPO_DIR / "MidiOutput" / "WebAPI"
PROFILE_FILE = REPO_DIR / ".colab_enabled_profiles"
SERVER_LOG = Path("/tmp/music_to_midi_web.log")
TUNNEL_LOG = Path("/tmp/music_to_midi_tunnel.log")


def _stop_proc(proc):
    if proc is None or proc.poll() is not None:
        return
    proc.terminate()
    try:
        proc.wait(timeout=5.0)
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait(timeout=5.0)


def _tail_log(path, limit=4000):
    if not path.is_file():
        return ""
    return path.read_text(encoding="utf-8", errors="replace")[-limit:]


for pattern in ("src.web_api.colab_server", "cloudflared tunnel"):
    subprocess.run(["pkill", "-f", pattern], check=False)
subprocess.run(["fuser", "-k", f"{PORT}/tcp"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1.0)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["MUSIC_TO_MIDI_ACCELERATOR"] = "cuda"
env["MPLBACKEND"] = "Agg"
env["WANDB_MODE"] = "disabled"
env["TF_CPP_MIN_LOG_LEVEL"] = "3"
env["MPLCONFIGDIR"] = "/tmp/matplotlib"
env["NUMBA_CACHE_DIR"] = "/tmp/numba_cache"
if PROFILE_FILE.is_file():
    enabled = PROFILE_FILE.read_text(encoding="utf-8").strip()
    if enabled:
        env["MUSIC_TO_MIDI_ENABLED_PROFILES"] = enabled
        env["MUSIC_TO_MIDI_REQUIRE_ENABLED_PROFILES"] = "1"

SERVER_LOG.unlink(missing_ok=True)
TUNNEL_LOG.unlink(missing_ok=True)

server_proc = None
tunnel_proc = None
try:
    with SERVER_LOG.open("w", encoding="utf-8") as server_log_handle:
        server_proc = subprocess.Popen(
            [
                str(VENV_PYTHON),
                "-m",
                "src.web_api.colab_server",
                "--host",
                "0.0.0.0",
                "--port",
                str(PORT),
                "--data-dir",
                str(DATA_DIR),
            ],
            cwd=str(REPO_DIR),
            env=env,
            stdin=subprocess.DEVNULL,
            stdout=server_log_handle,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

    health_url = f"http://127.0.0.1:{PORT}/api/v1/health"
    deadline = time.monotonic() + 60.0
    health_payload = None
    while time.monotonic() < deadline:
        if server_proc.poll() is not None:
            raise RuntimeError(
                f"Web server thoát sớm (code={server_proc.returncode}):\n{_tail_log(SERVER_LOG)}"
            )
        try:
            with urlopen(health_url, timeout=2.0) as resp:
                if resp.status == 200:
                    health_payload = json.loads(resp.read().decode("utf-8"))
                    if health_payload.get("status") == "ok":
                        break
        except (URLError, OSError, json.JSONDecodeError):
            time.sleep(0.5)

    if not health_payload or health_payload.get("status") != "ok":
        raise RuntimeError(
            f"Web server không phản hồi /api/v1/health trong 60 giây:\n{_tail_log(SERVER_LOG)}"
        )

    tunnel_url = None
    if PUBLIC_TUNNEL == "cloudflared":
        if shutil.which("cloudflared") is None:
            raise RuntimeError("Không tìm thấy cloudflared; hãy chạy lại Cell 2.")
        with TUNNEL_LOG.open("w", encoding="utf-8") as tunnel_log_handle:
            tunnel_proc = subprocess.Popen(
                ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
                stdin=subprocess.DEVNULL,
                stdout=tunnel_log_handle,
                stderr=subprocess.STDOUT,
                start_new_session=True,
            )

        tunnel_deadline = time.monotonic() + 30.0
        url_pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
        registered = False
        while time.monotonic() < tunnel_deadline:
            if tunnel_proc.poll() is not None:
                raise RuntimeError(
                    f"cloudflared thoát sớm (code={tunnel_proc.returncode}):\n{_tail_log(TUNNEL_LOG)}"
                )
            text = _tail_log(TUNNEL_LOG, limit=16000)
            if not tunnel_url:
                match = url_pattern.search(text)
                if match:
                    tunnel_url = match.group(0)
            if tunnel_url and "Registered tunnel connection" in text:
                registered = True
                break
            time.sleep(0.5)

        if not tunnel_url or not registered:
            raise RuntimeError(
                f"cloudflared không đăng ký được tunnel trong 30 giây:\n{_tail_log(TUNNEL_LOG)}"
            )

        verify_deadline = time.monotonic() + 15.0
        verified = False
        while time.monotonic() < verify_deadline:
            try:
                with urlopen(f"{tunnel_url}/api/v1/health", timeout=4.0) as resp:
                    if resp.status == 200:
                        verified = True
                        break
            except (URLError, OSError):
                time.sleep(1.5)
        if not verified:
            print(
                f"[Thông tin] Tunnel đã kết nối ({tunnel_url}), DNS trên Colab VM đang cập nhật — hãy mở trực tiếp từ trình duyệt của bạn.",
                flush=True,
            )

    colab_proxy_url = None
    if PUBLIC_TUNNEL == "colab_proxy_only":
        from google.colab import output

        colab_proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({PORT})")

    print("\n" + "=" * 68, flush=True)
    print("🎵 Music to MIDI Web UI đã sẵn sàng!", flush=True)
    print(f"• Local API Health : {health_url}", flush=True)
    if tunnel_url:
        print(f"• Web UI (Public)  : {tunnel_url}", flush=True)
        print(f"• OpenAPI Docs     : {tunnel_url}/docs", flush=True)
    if colab_proxy_url:
        print(f"• Colab Proxy URL  : {colab_proxy_url}", flush=True)
    print(f"• Log Server       : {SERVER_LOG}", flush=True)
    print("=" * 68 + "\n", flush=True)
except Exception:
    _stop_proc(tunnel_proc)
    _stop_proc(server_proc)
    raise

if KEEP_BLOCKING:
    print("Đang giữ server chạy và stream log (nhấn Stop để dừng)...", flush=True)
    try:
        with SERVER_LOG.open("r", encoding="utf-8", errors="replace") as log_stream:
            while True:
                if server_proc.poll() is not None:
                    raise RuntimeError(
                        f"Web server dừng bất ngờ (code={server_proc.returncode})"
                    )
                if tunnel_proc is not None and tunnel_proc.poll() is not None:
                    raise RuntimeError(
                        f"cloudflared dừng bất ngờ (code={tunnel_proc.returncode})"
                    )
                chunk = log_stream.read()
                if chunk:
                    print(chunk, end="", flush=True)
                time.sleep(1.5)
    except KeyboardInterrupt:
        print("\nĐang dừng Web UI và tunnel...", flush=True)
    finally:
        _stop_proc(tunnel_proc)
        _stop_proc(server_proc)
